upload dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/nycu-hw2-data.tar.gz /content/
!tar -zxf nycu-hw2-data.tar.gz

Phase 1: 224*448 with augmentation


In [ ]:
import os
import torch
import random
import numpy as np
import albumentations as A
from PIL import Image
from torch.utils.data import DataLoader
from tqdm import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from transformers import (
    RTDetrV2Config,
    RTDetrV2ForObjectDetection,
    RTDetrImageProcessor,
    get_cosine_schedule_with_warmup
)

# Configuration

TRAIN_IMG_DIR  = "nycu-hw2-data/train"
TRAIN_ANN_FILE = "nycu-hw2-data/train.json"
VAL_IMG_DIR  = "nycu-hw2-data/valid"
VAL_ANN_FILE  = "nycu-hw2-data/valid.json"
OUTPUT_DIR   = "./model_checkpoints"

TOTAL_EPOCHS = 30
BATCH_SIZE  = 32
SEED     = 42
TARGET_H   = 224
TARGET_W   = 448

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Augmentation

train_transform = A.Compose(
    [
        A.OneOf(
            [
                A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=1.0),
                A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=10, val_shift_limit=10, p=1.0),
                A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
            ],
            p=0.5,
        ),
        A.OneOf(
            [
                A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=1.0),
                A.GaussianBlur(blur_limit=(3, 5), p=1.0),
                A.MotionBlur(blur_limit=3, p=1.0),
            ],
            p=0.3,
        ),
        A.OneOf(
            [
                A.GaussNoise(std_range=(0.01, 0.08), per_channel=True, p=1.0),
                A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.3), p=1.0),
                A.ImageCompression(quality_range=(60, 100), p=1.0),
            ],
            p=0.3,
        ),
        A.CoarseDropout(
            num_holes_range=(4, 8),
            hole_height_range=(2, 4),
            hole_width_range=(2, 4),
            p=0.2
        ),
    ]

)

# Dataset & Collate

class DigitCocoDataset(torch.utils.data.Dataset):
    def __init__(self, img_folder, ann_file, transform=None):
        self.coco       = COCO(ann_file)
        self.ids        = list(sorted(self.coco.imgs.keys()))
        self.img_folder = img_folder
        self.transform  = transform

    def __getitem__(self, idx):
        img_id  = self.ids[idx]
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns    = self.coco.loadAnns(ann_ids)
        info    = self.coco.loadImgs(img_id)[0]
        img     = np.array(Image.open(os.path.join(self.img_folder, info["file_name"])).convert("RGB"))

        bboxes, labels = [], []
        for ann in anns:
            if ann["bbox"][2] > 0 and ann["bbox"][3] > 0:
                bboxes.append(ann["bbox"])
                labels.append(ann["category_id"] - 1)

        if self.transform:
            out    = self.transform(image=img, bboxes=bboxes, class_labels=labels)
            img    = out["image"]
            bboxes = list(out["bboxes"])
            labels = list(out["class_labels"])

        formatted = [
            {"image_id": img_id, "category_id": lbl, "bbox": list(bb),
             "area": bb[2] * bb[3], "iscrowd": 0}
            for bb, lbl in zip(bboxes, labels)
        ]
        return img, {"image_id": img_id, "annotations": formatted}, (img.shape[0], img.shape[1])

    def __len__(self):
        return len(self.ids)


def collate_fn(batch):
    images      = [item[0] for item in batch]
    annotations = [item[1] for item in batch]
    batch_dict  = processor(images=images, annotations=annotations, return_tensors="pt")
    batch_dict["image_ids"]  = [ann["image_id"] for ann in annotations]
    batch_dict["orig_sizes"] = torch.tensor([item[2] for item in batch])
    return batch_dict


processor = RTDetrImageProcessor.from_pretrained(
    "PekingU/rtdetr_v2_r50vd",
    size={ "width": TARGET_W, "height": TARGET_H},
    resample=3,
)

train_dataset = DigitCocoDataset(TRAIN_IMG_DIR, TRAIN_ANN_FILE, transform=train_transform)
valid_dataset = DigitCocoDataset(VAL_IMG_DIR,   VAL_ANN_FILE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=8, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=8, pin_memory=True)
coco_gt = COCO(VAL_ANN_FILE)

# Model & Optimizer

id2label = {i: str(i) for i in range(10)}
label2id = {v: k for k, v in id2label.items()}

config = RTDetrV2Config(
    backbone="timm/resnet50_gn.a1h_in1k",
    use_timm_backbone=True,
    use_pretrained_backbone=True,
    backbone_kwargs={"out_indices": (1, 2, 3), "features_only": True},
    num_labels=10,
    num_queries=200,
    encoder_layers=1,
    decoder_layers=6,
    id2label=id2label,
    label2id=label2id,
    auxiliary_loss=True,
    use_vfl_loss=True,
    num_denoising=50,
    prediction_bottleneck=True,
    matcher_class_cost=4.0,
    matcher_bbox_cost=8.0,
    matcher_giou_cost=4.0,
    weight_loss_vfl = 2.0,
    weight_loss_bbox = 8.0,
    weight_loss_giou = 3.0,
    dropout=0.2,
    activation_dropout=0.1,
)
model = RTDetrV2ForObjectDetection(config).to(device)

param_dicts = [
    {"params": [p for n, p in model.named_parameters() if "backbone" not in n], "lr": 5e-5},
    {"params": [p for n, p in model.named_parameters() if "backbone"    in n], "lr": 5e-6},
]
optimizer = torch.optim.AdamW(param_dicts, weight_decay=1e-3)

total_steps = TOTAL_EPOCHS * len(train_loader)

num_warmup_steps = 3 * len(train_loader)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=total_steps
)


# Eval helper

def run_eval(eval_model, loader):
    eval_model.eval()
    results = []
    val_loss_total = 0.0
    with torch.no_grad():
        for batch in tqdm(loader, desc="Eval", leave=False):
            pixel_values = batch["pixel_values"].to(device)
            labels       = [{k: v.to(device) for k, v in t.items()} for t in batch["labels"]]
            orig_sizes   = batch["orig_sizes"].to(device)
            outputs      = eval_model(pixel_values=pixel_values, labels = labels)
            val_loss_total  += outputs.loss.item()
            processed    = processor.post_process_object_detection(
                outputs, target_sizes=orig_sizes, threshold=0.01
            )
            for j, res in enumerate(processed):
                img_id = int(batch["image_ids"][j])
                for box, score, label in zip(res["boxes"], res["scores"], res["labels"]):
                    x1, y1, x2, y2 = box.tolist()
                    results.append({
                        "image_id":    img_id,
                        "category_id": int(label) + 1,
                        "bbox":        [x1, y1, x2 - x1, y2 - y1],
                        "score":       float(score),
                    })
    if not results:
        print("  [Warning] No detections.")
        return 0.0
    coco_dt   = coco_gt.loadRes(results)
    coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    print(f'val_loss:{(val_loss_total / len(loader)):.3f}')
    return coco_eval.stats[0]


# Training Loop

best_ap = 0.0

for epoch in range(TOTAL_EPOCHS):

    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{TOTAL_EPOCHS}")

    epoch_loss_total = 0.0
    epoch_vfl = 0.0
    epoch_bbox = 0.0
    epoch_giou = 0.0

    for batch_idx, batch in enumerate(pbar):
        pixel_values = batch["pixel_values"].to(device)
        labels       = [{k: v.to(device) for k, v in t.items()} for t in batch["labels"]]

        optimizer.zero_grad()
        outputs   = model(pixel_values=pixel_values, labels=labels)
        loss      = outputs.loss
        loss_dict = outputs.loss_dict
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)

        optimizer.step()
        scheduler.step()


        vfl_val  = loss_dict.get("loss_vfl",  torch.tensor(0.0)).item()
        bbox_val = loss_dict.get("loss_bbox", torch.tensor(0.0)).item()
        giou_val = loss_dict.get("loss_giou", torch.tensor(0.0)).item()

        batch_n = batch_idx + 1
        epoch_loss_total += loss.item()
        epoch_vfl += vfl_val
        epoch_bbox += bbox_val
        epoch_giou += giou_val
        pbar.set_postfix({
            "Avg_Tot": f"{epoch_loss_total / batch_n:.3f}",
            "Avg_VFL": f"{epoch_vfl / batch_n:.3f}",
            "Avg_BB":  f"{epoch_bbox / batch_n:.3f}",
            "Avg_G":   f"{epoch_giou / batch_n:.3f}",
        })

    # Validation
    if (epoch + 1) % 2 == 0 or epoch == 0:
        ap = run_eval(model, valid_loader)

        if ap > best_ap:
            best_ap = ap
            model.save_pretrained(os.path.join(OUTPUT_DIR, "best_model"))
            processor.save_pretrained(os.path.join(OUTPUT_DIR, "best_model"))




Phase 2: 256*512 without augmentation

In [ ]:
import os
import torch
import random
import numpy as np
from PIL import Image
from torch.utils.data import DataLoader
from tqdm import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from transformers import (
    RTDetrV2ForObjectDetection,
    RTDetrImageProcessor,
    get_cosine_schedule_with_warmup
)

#  Configuration

TRAIN_IMG_DIR  = "nycu-hw2-data/train"
TRAIN_ANN_FILE = "nycu-hw2-data/train.json"
VAL_IMG_DIR  = "nycu-hw2-data/valid"
VAL_ANN_FILE  = "nycu-hw2-data/valid.json"
OUTPUT_DIR   = "./model_checkpoints"

TOTAL_EPOCHS = 10
BATCH_SIZE  = 16
SEED     = 42
TARGET_H   = 256
TARGET_W   = 512

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Dataset & Collate

class DigitCocoDataset(torch.utils.data.Dataset):
    def __init__(self, img_folder, ann_file, transform=None):
        self.coco       = COCO(ann_file)
        self.ids        = list(sorted(self.coco.imgs.keys()))
        self.img_folder = img_folder
        self.transform  = transform

    def __getitem__(self, idx):
        img_id  = self.ids[idx]
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns    = self.coco.loadAnns(ann_ids)
        info    = self.coco.loadImgs(img_id)[0]
        img     = np.array(Image.open(os.path.join(self.img_folder, info["file_name"])).convert("RGB"))

        bboxes, labels = [], []
        for ann in anns:
            if ann["bbox"][2] > 0 and ann["bbox"][3] > 0:
                bboxes.append(ann["bbox"])
                labels.append(ann["category_id"] - 1)

        if self.transform:
            out    = self.transform(image=img, bboxes=bboxes, class_labels=labels)
            img    = out["image"]
            bboxes = list(out["bboxes"])
            labels = list(out["class_labels"])

        formatted = [
            {"image_id": img_id, "category_id": lbl, "bbox": list(bb),
             "area": bb[2] * bb[3], "iscrowd": 0}
            for bb, lbl in zip(bboxes, labels)
        ]
        return img, {"image_id": img_id, "annotations": formatted}, (img.shape[0], img.shape[1])

    def __len__(self):
        return len(self.ids)


def collate_fn(batch):
    images      = [item[0] for item in batch]
    annotations = [item[1] for item in batch]
    batch_dict  = processor(images=images, annotations=annotations, return_tensors="pt")
    batch_dict["image_ids"]  = [ann["image_id"] for ann in annotations]
    batch_dict["orig_sizes"] = torch.tensor([item[2] for item in batch])
    return batch_dict


processor = RTDetrImageProcessor.from_pretrained(
    "/content/model_checkpoints/best_model",
    size={ "width": TARGET_W, "height": TARGET_H},
    resample=3
)

train_dataset = DigitCocoDataset(TRAIN_IMG_DIR, TRAIN_ANN_FILE)
valid_dataset = DigitCocoDataset(VAL_IMG_DIR,   VAL_ANN_FILE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=8, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=8, pin_memory=True)
coco_gt = COCO(VAL_ANN_FILE)



# Model & Optimizer

model = RTDetrV2ForObjectDetection.from_pretrained("/content/model_checkpoints/best_model").to(device)
param_dicts = [
    {"params": [p for n, p in model.named_parameters() if "backbone" not in n], "lr": 5e-6},
    {"params": [p for n, p in model.named_parameters() if "backbone"    in n], "lr": 5e-7},
]
optimizer = torch.optim.AdamW(param_dicts, weight_decay=1e-3)

total_steps = TOTAL_EPOCHS * len(train_loader)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)


# Eval helper

def run_eval(eval_model, loader):
    eval_model.eval()
    results = []
    val_loss_total = 0.0
    with torch.no_grad():
        for batch in tqdm(loader, desc="Eval", leave=False):
            pixel_values = batch["pixel_values"].to(device)
            labels       = [{k: v.to(device) for k, v in t.items()} for t in batch["labels"]]
            orig_sizes   = batch["orig_sizes"].to(device)
            outputs      = eval_model(pixel_values=pixel_values, labels = labels)
            val_loss_total  += outputs.loss.item()
            processed    = processor.post_process_object_detection(
                outputs, target_sizes=orig_sizes, threshold=0.01
            )
            for j, res in enumerate(processed):
                img_id = int(batch["image_ids"][j])
                for box, score, label in zip(res["boxes"], res["scores"], res["labels"]):
                    x1, y1, x2, y2 = box.tolist()
                    results.append({
                        "image_id":    img_id,
                        "category_id": int(label) + 1,
                        "bbox":        [x1, y1, x2 - x1, y2 - y1],
                        "score":       float(score),
                    })
    if not results:
        print("  [Warning] No detections.")
        return 0.0
    coco_dt   = coco_gt.loadRes(results)
    coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    print(f'val_loss:{(val_loss_total / len(loader)):.3f}')
    return coco_eval.stats[0]


# Training Loop

best_ap = 0.0

for epoch in range(TOTAL_EPOCHS):

    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{TOTAL_EPOCHS}")

    epoch_loss_total = 0.0
    epoch_vfl = 0.0
    epoch_bbox = 0.0
    epoch_giou = 0.0

    for batch_idx, batch in enumerate(pbar):
        pixel_values = batch["pixel_values"].to(device)
        labels       = [{k: v.to(device) for k, v in t.items()} for t in batch["labels"]]

        optimizer.zero_grad()
        outputs   = model(pixel_values=pixel_values, labels=labels)
        loss      = outputs.loss
        loss_dict = outputs.loss_dict
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)

        optimizer.step()
        scheduler.step()


        vfl_val  = loss_dict.get("loss_vfl",  torch.tensor(0.0)).item()
        bbox_val = loss_dict.get("loss_bbox", torch.tensor(0.0)).item()
        giou_val = loss_dict.get("loss_giou", torch.tensor(0.0)).item()

        batch_n = batch_idx + 1
        epoch_loss_total += loss.item()
        epoch_vfl += vfl_val
        epoch_bbox += bbox_val
        epoch_giou += giou_val
        pbar.set_postfix({
            "Avg_Tot": f"{epoch_loss_total / batch_n:.3f}",
            "Avg_VFL": f"{epoch_vfl / batch_n:.3f}",
            "Avg_BB":  f"{epoch_bbox / batch_n:.3f}",
            "Avg_G":   f"{epoch_giou / batch_n:.3f}",
        })

    # Validation

    ap = run_eval(model, valid_loader)

    if ap > best_ap:
        best_ap = ap
        model.save_pretrained(os.path.join(OUTPUT_DIR, "best_model"))
        processor.save_pretrained(os.path.join(OUTPUT_DIR, "best_model"))




Inference

In [ ]:
import torch
import json
import os
import glob
import re
from PIL import Image
from tqdm import tqdm
from transformers import (
    RTDetrV2ForObjectDetection,
    RTDetrImageProcessor,
)

image_folder = "nycu-hw2-data/test"
model_path = "/content/model_checkpoints/best_model"
output_json = "pred.json"
threshold = 0.0025

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# LOAD MODEL & PROCESSOR
processor = RTDetrImageProcessor.from_pretrained(
  model_path
)

model = RTDetrV2ForObjectDetection.from_pretrained(model_path).to(device)
model.eval()

# GET SORTED IMAGES
image_paths = []
for ext in ('*.png', '*.jpg', '*.jpeg'):
    image_paths.extend(glob.glob(os.path.join(image_folder, ext)))

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]

image_paths.sort(key=natural_sort_key)

# RUN INFERENCE & COLLECT RESULTS
coco_results = []

print(f"Generating predictions for {len(image_paths)} images...")

with torch.no_grad():
    for img_path in tqdm(image_paths):
        image_id = int(re.search(r'(\d+)', os.path.basename(img_path)).group(1))
        image = Image.open(img_path).convert("RGB")
        width, height = image.size
        inputs = processor(images=image, return_tensors="pt").to(device)

        outputs = model(**inputs)

        target_sizes = torch.tensor([[height, width]])
        results = processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=threshold)[0]

        image_results = []
        for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
            xmin, ymin, xmax, ymax = box.tolist()

            coco_bbox = [xmin, ymin, xmax - xmin, ymax - ymin]

            image_results.append({
                        "image_id": image_id,
                        "bbox": coco_bbox,
                        "score": score.item(),
                        "category_id": int(label.item()) + 1
                      })

        image_results = sorted(image_results, key=lambda x: x["score"], reverse=True)[:100]
        coco_results.extend(image_results)

# SAVE TO FILE
with open(output_json, "w") as f:
    json.dump(coco_results, f, indent=4)

print(f"Successfully saved {len(coco_results)} detections to {output_json}")


